# Notebook 01 - Data Loading & Visualisation

**Project:** ΛCDM fitting with WMAP9 + DESI  
**Author:** Cherian P Ittyipe

---

## Purpose

This is the **first notebook** in the pipeline. The only goals here are:

1. Load the three observational datasets
2. Verify each file is readable and has the correct shape
3. Plot each dataset as a sanity check before any fitting begins

> **No likelihood evaluation. No fitting. No modelling.**  
> Those happen in later notebooks.

---

## Datasets used in this notebook

| Dataset | File | Purpose |
|---------|------|---------|
| WMAP9 TT | `wmap_tt_spectrum_9yr_v5.txt` | CMB power spectrum — plot |
| Planck 2018 TT | `COM_PowerSpect_CMB-TT-full_R3.01.txt` | Comparison plot only |
| DESI DR1 BAO | 14 files in `bao_data/` | BAO measurements - plot |

---



In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os, sys

# ── Base paths ────────────────────────────────────────────────────────────────
BASE    = "/Users/cherianpi/Desktop/WMAP+DESI"
BAO_DIR = BASE + "/bao_data"

print("BASE    :", BASE)
print("BAO_DIR :", BAO_DIR)
print("Imports OK")

---

## Section 1 — WMAP9 CMB TT Power Spectrum

### What this file contains

File: `wmap_tt_spectrum_9yr_v5.txt`  
Source: NASA LAMBDA

This is the **WMAP 9-year binned TT band-power spectrum** — the main observable  
we use from WMAP for CMB constraints.

### Column layout

| Index | Variable | Description |
|-------|----------|-------------|
| `[:,0]` | `ell_min` | Lower edge of ℓ bin |
| `[:,1]` | `ell_max` | Upper edge of ℓ bin |
| `[:,2]` | `D_ell` | D_ℓ = ℓ(ℓ+1)Cℓ/2π  [μK²] |
| `[:,3]` | `err_up` | Upper 1σ error bar |
| `[:,4]` | `err_down` | Lower 1σ error bar (positive number) |

Bin centres: `ell = (ell_min + ell_max) / 2`


In [ ]:
# ── Load WMAP9 TT spectrum ────────────────────────────────────────────────────
WMAP_FILE = BASE + "/wmap_tt_spectrum_9yr_v5.txt"

if not os.path.exists(WMAP_FILE):
    print("ERROR: WMAP9 file not found at:", WMAP_FILE); sys.exit(1)

wmap = np.loadtxt(WMAP_FILE, comments="#")
print("Shape  :", wmap.shape)          # expect (1199, 5)
print("ℓ range:", wmap[:,0].min(), "–", wmap[:,1].max())
print("First 3 rows:")
print(wmap[:3])

# ── Parse columns ────────────────────────────────────────────────────────────
wmap_ell      = (wmap[:,0] + wmap[:,1]) / 2.0   # bin centre
wmap_Dell     = wmap[:,2]                        # D_ℓ [μK²]
wmap_err_up   = wmap[:,3]
wmap_err_down = wmap[:,4]                        # positive number

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

ax.errorbar(wmap_ell, wmap_Dell,
            yerr=[wmap_err_down, wmap_err_up],
            fmt='none', ecolor='#2980B9', elinewidth=1.0, alpha=0.8)
ax.scatter(wmap_ell, wmap_Dell, s=14, color='#2980B9', label='WMAP9 TT', zorder=4)

ax.set_xlabel(r'Multipole $\ell$', fontsize=13)
ax.set_ylabel(r'$D_\ell \;[\mu\mathrm{K}^2]$', fontsize=13)
ax.set_title('WMAP9 CMB TT Power Spectrum', fontsize=14)
ax.set_xlim(0, 1200)
ax.legend(fontsize=11)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.tick_params(which='both', direction='in')

plt.tight_layout()
plt.savefig(BASE + "/Images/wmap9_tt_only.png", dpi=150)
plt.show()
print("Saved: Images/wmap9_tt_only.png")

---

## Section 2 — WMAP9 + Planck 2018 TT Comparison

### What this section does

Loads the Planck 2018 TT spectrum and overlays it with WMAP9 to confirm:  
- Both are in the same units (μK²) — no unit mismatch  
- They are consistent in the overlap region ℓ ≤ 1000  
- Planck extends to higher ℓ where WMAP has no data  

> **Planck is used here for visual comparison only.**  
> The actual fitting uses WMAP9 likelihood, not Planck.

### Planck 2018 column layout

File: `COM_PowerSpect_CMB-TT-full_R3.01.txt`  
Source: Planck Legacy Archive

| Index | Variable | Description |
|-------|----------|-------------|
| `[:,0]` | `ell` | Multipole |
| `[:,1]` | `D_ell` | D_ℓ [μK²] |
| `[:,2]` | `err_minus` | Lower 1σ (positive) |
| `[:,3]` | `err_plus` | Upper 1σ |

### Pass criteria
- Planck shape must be `(2507, 4)`  
- Both datasets peak at ℓ ≈ 200 at the same D_ℓ value  
- No vertical offset between them in the overlap region

In [ ]:
# ── Load Planck 2018 TT ───────────────────────────────────────────────────────
PLANCK_FILE = BASE + "/COM_PowerSpect_CMB-TT-full_R3.01.txt"

if not os.path.exists(PLANCK_FILE):
    print("ERROR: Planck file not found at:", PLANCK_FILE); sys.exit(1)

planck = np.loadtxt(PLANCK_FILE, comments="#")
print("Planck shape  :", planck.shape)         # expect (2507, 4)
print("Planck ℓ range:", planck[:,0].min(), "–", planck[:,0].max())

planck_ell      = planck[:,0]
planck_Dell     = planck[:,1]
planck_err_down = planck[:,2]
planck_err_up   = planck[:,3]

# ── Figure 1: Full ℓ range (0–2600) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.errorbar(planck_ell, planck_Dell,
            yerr=[planck_err_down, planck_err_up],
            fmt='none', ecolor='#C0392B', elinewidth=0.5, alpha=0.6)
ax.scatter(planck_ell, planck_Dell, s=3,  color='#C0392B', label='Planck 2018', zorder=3)

ax.errorbar(wmap_ell, wmap_Dell,
            yerr=[wmap_err_down, wmap_err_up],
            fmt='none', ecolor='#2980B9', elinewidth=1.0, alpha=0.8)
ax.scatter(wmap_ell, wmap_Dell, s=18, color='#2980B9', label='WMAP9',       zorder=4)

ax.set_xlabel(r'Multipole $\ell$', fontsize=13)
ax.set_ylabel(r'$D_\ell \;[\mu\mathrm{K}^2]$', fontsize=13)
ax.set_title('CMB TT Power Spectrum: WMAP9 vs Planck 2018', fontsize=14)
ax.set_xlim(0, 2600)
ax.legend(fontsize=11, markerscale=3)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.tick_params(which='both', direction='in')
plt.tight_layout()
plt.savefig(BASE + "/Images/tt_spectrum_wmap9_planck.png", dpi=150)
plt.show()
print("Saved: Images/tt_spectrum_wmap9_planck.png")

# ── Figure 2: Zoom ℓ ≤ 1000 (overlap region) ─────────────────────────────────
mask = planck_ell <= 1000

fig, ax = plt.subplots(figsize=(12, 5))

ax.errorbar(planck_ell[mask], planck_Dell[mask],
            yerr=[planck_err_down[mask], planck_err_up[mask]],
            fmt='none', ecolor='#C0392B', elinewidth=0.8, alpha=0.7)
ax.scatter(planck_ell[mask], planck_Dell[mask], s=8,  color='#C0392B', label='Planck 2018', zorder=3)

ax.errorbar(wmap_ell, wmap_Dell,
            yerr=[wmap_err_down, wmap_err_up],
            fmt='none', ecolor='#2980B9', elinewidth=1.5, alpha=0.8)
ax.scatter(wmap_ell, wmap_Dell, s=20, color='#2980B9', label='WMAP9',       zorder=4)

ax.set_xlabel(r'Multipole $\ell$', fontsize=13)
ax.set_ylabel(r'$D_\ell \;[\mu\mathrm{K}^2]$', fontsize=13)
ax.set_title(r'CMB TT Power Spectrum: WMAP9 vs Planck 2018 ($\ell \leq 1000$)', fontsize=14)
ax.set_xlim(0, 1050)
ax.legend(fontsize=11, markerscale=2)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.tick_params(which='both', direction='in')
plt.tight_layout()
plt.savefig(BASE + "/Images/tt_spectrum_wmap9_planck_zoom.png", dpi=150)
plt.show()
print("Saved: Images/tt_spectrum_wmap9_planck_zoom.png")

---

## Section 3 — DESI DR1 BAO Measurements

### What DESI measures

DESI measures the **Baryon Acoustic Oscillation scale** — a known physical  
ruler — at 7 different redshifts across different galaxy/quasar populations.  
It reports two ratios at each redshift:

- **D_M / r_d** — comoving angular diameter distance / sound horizon  
- **D_H / r_d** — Hubble distance (= c/H(z)) / sound horizon  

These ratios are what we compare our ΛCDM model predictions against.

### The 7 redshift bins

| Label | Tracer | z_eff | z range | Measures |
|-------|--------|-------|---------|----------|
| BGS | Bright galaxies | 0.295 | 0.1–0.4 | D_V/r_d (isotropic) |
| LRG1 | Luminous red galaxies | 0.510 | 0.4–0.6 | D_M/r_d, D_H/r_d |
| LRG2 | Luminous red galaxies | 0.706 | 0.6–0.8 | D_M/r_d, D_H/r_d |
| LRG3+ELG1 | LRG + Emission line | 0.930 | 0.8–1.1 | D_M/r_d, D_H/r_d |
| ELG2 | Emission line galaxies | 1.317 | 1.1–1.6 | D_M/r_d, D_H/r_d |
| QSO | Quasars | 1.491 | 0.8–2.1 | D_M/r_d, D_H/r_d |
| Lya | Lyman-α forest | 2.330 | 1.77–4.16 | D_M/r_d, D_H/r_d |

### File format

Each tracer has two files:
- `_mean.txt` — one row: `[z_eff, DM_over_rd, DH_over_rd]`  
  (BGS has `[z_eff, DV_over_rd]` — isotropic only)
- `_cov.txt`  — 2×2 covariance matrix (1×1 for BGS)


In [ ]:
# ── DESI DR1 file stems and modes ─────────────────────────────────────────────
TRACERS = {
    "BGS":       ("desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4",  "iso"),
    "LRG1":      ("desi_2024_gaussian_bao_LRG_GCcomb_z0.4-0.6",               "aniso"),
    "LRG2":      ("desi_2024_gaussian_bao_LRG_GCcomb_z0.6-0.8",               "aniso"),
    "LRG3+ELG1": ("desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1", "aniso"),
    "ELG2":      ("desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6",     "aniso"),

    # QSO is DV/rd only in your file, so it is isotropic
    "QSO":       ("desi_2024_gaussian_bao_QSO_GCcomb_z0.8-2.1",               "iso"),

    "LYA":       ("desi_2024_gaussian_bao_Lya_GCcomb",                        "aniso"),
}

# ── Load all 14 files ─────────────────────────────────────────────────────────
desi = {}

print(f"{'Tracer':<12} {'z_eff':>6}  {'Values'}")
print("-" * 55)

for label, (stem, mode) in TRACERS.items():
    mean_path = f"{BAO_DIR}/{stem}_mean.txt"
    cov_path  = f"{BAO_DIR}/{stem}_cov.txt"

    if not os.path.exists(mean_path):
        print(f"ERROR: {mean_path} not found")
        continue

    if not os.path.exists(cov_path):
        print(f"ERROR: {cov_path} not found")
        continue

    mean = load_desi_mean(mean_path, mode)
    cov  = np.atleast_2d(np.loadtxt(cov_path, comments="#"))

    desi[label] = {
        "mean": mean,
        "cov": cov,
        "mode": mode
    }

    vals = "  ".join([f"{v:.4f}" for v in mean[1:]])
    print(f"{label:<12} {mean[0]:>6.3f}  {vals}")

print("-" * 55)
print(f"Loaded {len(desi)}/7 tracers successfully.")
# ── Plot: D_M/r_d and D_H/r_d vs redshift ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

COLORS = {
    "BGS":       "#8E44AD",
    "LRG1":      "#27AE60",
    "LRG2":      "#1E8449",
    "LRG3+ELG1": "#148F77",
    "ELG2":      "#117A65",
    "QSO":       "#E67E22",
    "LYA":       "#C0392B",
}

for label, d in desi.items():
    m   = d["mean"]
    cov = d["cov"]
    z   = m[0]
    c   = COLORS[label]

    if d["mode"] == "iso":
        # BGS: D_V/r_d — plot on D_M panel as square marker
        axes[0].errorbar(z, m[1], yerr=np.sqrt(float(cov)),
                         fmt='s', color=c, capsize=5, ms=8, label=f"{label} (DV)")
    else:
        err_DM = np.sqrt(cov[0, 0])
        err_DH = np.sqrt(cov[1, 1])
        axes[0].errorbar(z, m[1], yerr=err_DM,
                         fmt='o', color=c, capsize=5, ms=7, label=label)
        axes[1].errorbar(z, m[2], yerr=err_DH,
                         fmt='o', color=c, capsize=5, ms=7, label=label)

for ax, ylabel, title in zip(
    axes,
    [r'$D_M / r_d$',               r'$D_H / r_d$'],
    ['Comoving Distance / r_d',    'Hubble Distance / r_d']
):
    ax.set_xlabel(r'Redshift $z$', fontsize=13)
    ax.set_ylabel(ylabel,          fontsize=13)
    ax.set_title(f'DESI DR1 BAO — {title}', fontsize=13)
    ax.legend(fontsize=9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(which='both', direction='in')

plt.tight_layout()
plt.savefig(BASE + "/Images/desi_dr1_bao_measurements.png", dpi=150)
plt.show()
print("Saved: Images/desi_dr1_bao_measurements.png")

---

## ✅ Notebook 01 — Pass Checklist

Before moving to Notebook 02, confirm all of the following:

**WMAP9**
- [ ] Shape is `(1199, 5)`
- [ ] First acoustic peak visible at ℓ ≈ 200, D_ℓ ≈ 5500–6000 μK²
- [ ] `Images/wmap9_tt_only.png` saved

**Planck comparison**
- [ ] Shape is `(2507, 4)`
- [ ] WMAP9 and Planck consistent in the ℓ ≤ 1000 overlap region
- [ ] `Images/tt_spectrum_wmap9_planck.png` saved
- [ ] `Images/tt_spectrum_wmap9_planck_zoom.png` saved

**DESI BAO**
- [ ] All 7/7 tracers loaded with no errors
- [ ] z_eff values: 0.295, 0.510, 0.706, 0.930, 1.317, 1.491, 2.330
- [ ] D_M/r_d increases monotonically with redshift
- [ ] `Images/desi_dr1_bao_measurements.png` saved

---

**Next → Notebook 02 — WMAP9 Likelihood Setup**

In Notebook 02 we will compile the WMAP9 Fortran likelihood,  
wrap it in Python, and verify it returns the correct χ² for a test model.